In [3]:

# !pip install transformers torch jiwer pingouin statsmodels seaborn

import json, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
import jiwer
import pingouin as pg

In [4]:
# !git clone https://github.com/ctbaumler/personal_style_postedit

rows = []
for file in glob.glob("personal_style_postedit/logs/*.json"):
    data = json.load(open(file))
    pid = data["user_info"]["id"]

    for task in data["responses"].values():
        if "scenario" not in task: 
            continue
        rows.append({
            "pid": pid,
            "scenario": task["scenario"],
            "type": "treatment" if task["model_generation_shown"] == 1 else "control",
            "draft": task["model_generation"],
            "final": task["final_version"],
        })

df = pd.DataFrame(rows)
print(df.shape)
print(df["type"].value_counts())

(486, 5)
type
treatment    324
control      162
Name: count, dtype: int64


In [5]:
first_control = df[df["type"] == "control"].iloc[0]

print(first_control)

pid                      03acc614-1708-42a6-b142-9ef7a0f0ce4e
scenario                                           condolence
type                                                  control
draft       Dear J,\n\nMy deepest condolences for your los...
final       Dear J,\n\nMy deepest condolences to you and y...
Name: 4, dtype: object


- treatment means the person post-edited an LLM draft.
- control means they wrote it themselves.

In [6]:
def clean(text):
    return re.sub(r"[^\w\s']", "", text.lower())

df["n_words"] = df["final"].apply(lambda t: len(t.split()))

t = df[df["type"] == "treatment"].copy()      # only post-edited texts
t["wer"] = [jiwer.wer(clean(d), clean(f)) for d, f in zip(t["draft"], t["final"])]
t["words_changed"] = t["wer"] * t["draft"].apply(lambda x: len(x.split()))

print(t[["n_words", "wer", "words_changed"]].describe())

          n_words         wer  words_changed
count  324.000000  324.000000     324.000000
mean   159.703704    0.177246      26.656758
std     22.381270    0.256964      38.669908
min     96.000000    0.000000       0.000000
25%    150.000000    0.020000       3.000000
50%    156.000000    0.095895      14.000000
75%    166.000000    0.197959      29.250000
max    348.000000    2.113333     319.113333


##### WER =(number of word substitutions + insertions + deletions)
-------------------------------------------------------
number of words in reference

- n_words is the length of the final text.
- wer is the fraction of the draft that was rewritten (0 means untouched, 1 means fully rewritten).
- words_changed is the same thing as a raw count.

In [7]:
# !pip install einops

In [8]:
import os

In [9]:
print(os.getenv("HF_TOKEN") is not None)

False


In [10]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Set the correct, existing repo name
model_path = "rrivera1849/LUAR-CRUD"
hf_token = os.getenv("HF_TOKEN")

# 2. Pass the variable into the load methods instead of the old string
tokenizer = AutoTokenizer.from_pretrained(model_path, token=hf_token)
model = AutoModel.from_pretrained(model_path, trust_remote_code=True, token=hf_token)

model.eval()



LUAR(
  (transformer): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNor

In [11]:


def embed(text):
    words = text.split()
    chunks = [" ".join(words[i:i+60]) for i in range(0, len(words), 60)]   # pieces of 60 words
    enc = tokenizer(chunks, max_length=128, padding="max_length",
                    truncation=True, return_tensors="pt")
    ids = enc["input_ids"].unsqueeze(0)          # LUAR wants shape (1, pieces, tokens)
    mask = enc["attention_mask"].unsqueeze(0)
    with torch.no_grad():
        vec = model(input_ids=ids, attention_mask=mask)
    return vec[0].numpy()                         # one vector (512 numbers) per document

### Compute the similarity scores

In [ ]:
# vectors for every text (takes a few minutes)
df["final_vec"] = df["final"].apply(embed)
t["final_vec"] = df.loc[t.index, "final_vec"]
t["draft_vec"] = t["draft"].apply(embed)

def cos(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# each person's "own style" = average vector of their 2 control texts
profiles = {}
for pid, g in df[df["type"] == "control"].groupby("pid"):
    profiles[pid] = np.mean(list(g["final_vec"]), axis=0)

t["self_sim"] = [cos(v, profiles[p]) for v, p in zip(t["final_vec"], t["pid"])]
t["llm_sim"]  = [cos(f, d) for f, d in zip(t["final_vec"], t["draft_vec"])]

print(t[["self_sim", "llm_sim"]].describe())

In [ ]:
for y in ["self_sim", "llm_sim"]:
    for x in ["n_words", "words_changed", "wer"]:
        r, p = stats.pearsonr(t[x], t[y])
        rho, sp = stats.spearmanr(t[x], t[y])
        print(f"{y} vs {x}:  Pearson r={r:.3f} (p={p:.3f}),  Spearman rho={rho:.3f} (p={sp:.3f})")

### Conclusion

The correlation analysis shows that editing amount is strongly associated with the similarity of the final text to the original LLM draft and moderately associated with similarity to the participant's own writing style.

* **Self-similarity** shows a positive relationship with both document length and editing amount. In particular, `words_changed` and `WER` have moderate positive correlations with self-similarity (`r ≈ 0.36`), suggesting that greater editing is associated with the final text becoming more similar to the participant's own style.
* **LLM-similarity** shows negative relationships with document length and editing amount. The strongest effects are observed for `words_changed` and `WER` (`r ≈ -0.82`, Spearman `ρ ≈ -0.86`), indicating that heavily edited texts tend to become substantially less similar to the original LLM-generated draft.
* The similar Pearson and Spearman results for most comparisons indicate that these relationships are generally consistent in both their numerical and rank-based patterns.

Overall, the results provide preliminary evidence that **greater post-editing is associated with movement away from the LLM's stylistic representation and toward the participant's own stylistic representation**. However, these are bivariate correlations and do not establish causality. Since multiple observations come from the same participants, repeated-measures correlation and regression analyses are required for a more appropriate within-participant analysis.


In [ ]:
for y in ["self_sim", "llm_sim"]:
    for x in ["n_words", "wer"]:
        result = pg.rm_corr(data=t, x=x, y=y, subject="pid")
        print(y, "vs", x, "-> r =", round(result["r"].iloc[0], 3), " p =", round(result["pval"].iloc[0], 4))

### Conclusion

Repeated-measures correlation shows that **editing amount (WER) is strongly associated with stylistic shift**: more editing is linked to higher self-similarity (`r = 0.316, p < .001`) and lower LLM-similarity (`r = -0.700, p < .001`). Document length has only a small relationship with self-similarity and no significant relationship with LLM-similarity. Overall, **editing amount appears to be a stronger predictor of stylistic change than document length**.


In [ ]:
model_fit = smf.ols("self_sim ~ n_words + wer + C(scenario)", data=t).fit(
    cov_type="cluster", cov_kwds={"groups": t["pid"]})
print(model_fit.summary())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

sns.regplot(data=t, x="n_words", y="self_sim", ax=axes[0, 0])
sns.regplot(data=t, x="wer",     y="self_sim", ax=axes[0, 1])
sns.regplot(data=t, x="n_words", y="llm_sim",  ax=axes[1, 0])
sns.regplot(data=t, x="wer",     y="llm_sim",  ax=axes[1, 1])

plt.tight_layout()
plt.show()

In [ ]:
long_docs = t[t["n_words"] >= 100]     # keep only texts with at least 100 words

for k in [20, 40, 60, 80, 100]:
    sims = []
    for text, pid in zip(long_docs["final"], long_docs["pid"]):
        short_text = " ".join(text.split()[:k])        # keep only the first k words
        sims.append(cos(embed(short_text), profiles[pid]))
    print(k, "words -> average self_sim =", round(np.mean(sims), 4))

print("full text -> average self_sim =", round(long_docs["self_sim"].mean(), 4))

In [16]:
t2 = t[t["n_words"] < 300]

model_fit2 = smf.ols("self_sim ~ n_words + wer + C(scenario)", data=t2).fit(
    cov_type="cluster", cov_kwds={"groups": t2["pid"]})
print(model_fit2.params[["n_words", "wer"]])
print(model_fit2.pvalues[["n_words", "wer"]])

n_words    0.000242
wer        0.105000
dtype: float64
n_words    1.857690e-01
wer        7.405100e-10
dtype: float64


In [17]:
profile_cache = {}

for k in [20, 40, 60, 80, 100]:
    sims = []
    for text, pid in zip(long_docs["final"], long_docs["pid"]):
        if (pid, k) not in profile_cache:
            ctrl = df[(df["pid"] == pid) & (df["type"] == "control")]["final"]
            vecs = [embed(" ".join(c.split()[:k])) for c in ctrl]
            profile_cache[(pid, k)] = np.mean(vecs, axis=0)
        short_vec = embed(" ".join(text.split()[:k]))
        sims.append(cos(short_vec, profile_cache[(pid, k)]))
    print(k, "words -> average self_sim =", round(np.mean(sims), 4))

20 words -> average self_sim = 0.6087
40 words -> average self_sim = 0.6401
60 words -> average self_sim = 0.6419
80 words -> average self_sim = 0.7274
100 words -> average self_sim = 0.726


### Conclusion

The text-length sensitivity analysis shows that self-similarity increases as more text is provided to LUAR, rising from `0.6087` at 20 words to `0.7274` at 80 words, after which it stabilizes. This suggests that LUAR similarity scores are somewhat sensitive to document length, particularly for shorter texts. Therefore, the small association between document length and self-similarity should be interpreted cautiously, while the stronger relationship between editing amount (WER) and self-similarity remains more compelling.


In [18]:
def embed(text, chunk=60):
    words = text.split()
    chunks = [" ".join(words[i:i+chunk]) for i in range(0, len(words), chunk)]
    enc = tokenizer(chunks, max_length=128, padding="max_length",
                    truncation=True, return_tensors="pt")
    ids = enc["input_ids"].unsqueeze(0)
    mask = enc["attention_mask"].unsqueeze(0)
    with torch.no_grad():
        vec = model(input_ids=ids, attention_mask=mask)
    return vec[0].numpy()

# rerun the fair truncation test with 30-word pieces

cache30 = {}
for k in [20, 40, 60, 80, 100]:
    sims = []
    for text, pid in zip(long_docs["final"], long_docs["pid"]):
        if (pid, k) not in cache30:
            ctrl = df[(df["pid"] == pid) & (df["type"] == "control")]["final"]
            vecs = [embed(" ".join(c.split()[:k]), chunk=30) for c in ctrl]
            cache30[(pid, k)] = np.mean(vecs, axis=0)
        short_vec = embed(" ".join(text.split()[:k]), chunk=30)
        sims.append(cos(short_vec, cache30[(pid, k)]))
    print(k, "words -> average self_sim =", round(np.mean(sims), 4))

20 words -> average self_sim = 0.6087
40 words -> average self_sim = 0.7168
60 words -> average self_sim = 0.7184
80 words -> average self_sim = 0.7622
100 words -> average self_sim = 0.7821


## conclusion

Within the natural length range of the dataset (about 150-170 words), document length had no clear independent effect on self-similarity after controlling for edit amount and scenario (β = 0.0002, p = .19), whereas edit amount remained a strong predictor (β = 0.105, p < .001). A truncation experiment showed that similarity rose from 0.61 at 20 words to about 0.73-0.78 at 100 words, with the rise concentrated where the input grows from one to two chunks, indicating that LUAR scores depend on the number of input segments rather than on length per se. Similarity scores for short spans are therefore not directly comparable with document-level scores.